# Chapter 3 Lab — Morphology and Word Vectors

Part A: stemming vs. lemmatization. Part B: co-occurrence vectors built by hand, then compared
against a small pretrained embedding model. Uses only NLTK's bundled corpora (no external
download of copyrighted text).

In [ ]:
import nltk
for pkg in ["punkt", "wordnet", "omw-1.4", "gutenberg", "averaged_perceptron_tagger"]:
    nltk.download(pkg, quiet=True)

## Part A — Stemming vs. Lemmatization

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer

words = ["studies", "studying", "played", "better", "geese", "ponies"]
porter, snowball, lancaster, lemm = PorterStemmer(), SnowballStemmer("english"), LancasterStemmer(), WordNetLemmatizer()

for w in words:
    print(f"{w:10s} porter={porter.stem(w):10s} snowball={snowball.stem(w):10s} "
          f"lancaster={lancaster.stem(w):10s} lemma={lemm.lemmatize(w, pos='v')}")

## Part B — Co-occurrence matrix from scratch

In [ ]:
import numpy as np
from nltk.corpus import gutenberg

tokens = [w.lower() for w in gutenberg.words("carroll-alice.txt") if w.isalpha()][:20000]
vocab = sorted(set(tokens))
idx = {w: i for i, w in enumerate(vocab)}
window = 4
M = np.zeros((len(vocab), len(vocab)), dtype=np.float32)
for i, w in enumerate(tokens):
    for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
        if i != j:
            M[idx[w], idx[tokens[j]]] += 1
print("Co-occurrence matrix shape:", M.shape)

In [ ]:
def nearest(word, k=5):
    v = M[idx[word]]
    sims = M @ v / (np.linalg.norm(M, axis=1) * np.linalg.norm(v) + 1e-8)
    top = np.argsort(-sims)[1:k+1]
    return [vocab[i] for i in top]

print(nearest("alice"))
print(nearest("queen"))

## Part C — Compare against a pretrained embedding model

In [ ]:
try:
    import gensim.downloader as api
    model = api.load("glove-wiki-gigaword-50")  # small pretrained model
    print(model.most_similar("queen", topn=5))
    print(model.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
except Exception as e:
    print("gensim model download unavailable offline — see README.", e)

## Exercise

Compare `nearest("alice")` from the hand-built co-occurrence matrix against
`model.most_similar("alice")` from the pretrained embedding (if available). Which neighbours
feel more semantically coherent, and why might that be?